In [11]:
"""
修复SHAP版本兼容性问题：移除不支持的'ax'参数，适配旧版SHAP
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.utils import shuffle
import xgboost as xgb
import lightgbm as lgb
import shap
import warnings
warnings.filterwarnings('ignore')

# 基础配置（确保中文显示和图片清晰度）
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']  # 增加备选字体，避免中文显示异常
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150  # 提高图片默认分辨率
plt.rcParams['savefig.dpi'] = 150  # 确保保存图片的分辨率

def curve_poly(x, y, degree):
    """多项式拟合工具函数"""
    mask = ~np.isnan(x) & ~np.isnan(y)
    x_clean, y_clean = x[mask], y[mask]
    if len(x_clean) < degree + 1:
        degree = min(degree, len(x_clean) - 1)
        if degree < 1: 
            print(f"拟合警告：{len(x_clean)}个有效样本，仅支持1阶以下拟合")
            return x_clean, y_clean
    try:
        coeffs = np.polyfit(x_clean, y_clean, deg=degree)
        poly = np.poly1d(coeffs)
        line_x = np.linspace(min(x_clean), max(x_clean), 200)
        line_y = poly(line_x)
        return line_x, line_y
    except Exception as e:
        print(f"拟合失败：{str(e)}，返回原始点")
        return x_clean, y_clean

def explore_data(name, df, save_prefix):
    """数据探索可视化（确保图片保存）"""
    print(f"\n{'='*60}")
    print(f"数据探索: {name}")
    print(f"{'='*60}")
    
    # 明确特征与目标变量（避免因列顺序导致错误）
    if 'RASTERVALU' not in df.columns:
        raise ValueError(f"数据{name}中未找到目标变量'RASTERVALU'，请检查数据列名")
    X = df.drop(columns=['RASTERVALU'])
    y = df['RASTERVALU']
    print(f"特征数量: {X.shape[1]}, 样本数量: {X.shape[0]}, 特征列表: {list(X.columns)}")
    
    # 图1: 目标变量分布
    fig1, ax1 = plt.subplots(figsize=(8, 6))
    ax1.hist(y, bins=30, color='steelblue', edgecolor='white', alpha=0.7)
    ax1.axvline(y.mean(), color='red', linestyle='--', label=f'均值={y.mean():.2f}')
    ax1.set_xlabel('RASTERVALU（目标变量）')
    ax1.set_ylabel('频数')
    ax1.set_title(f'{name} - 目标变量分布')
    ax1.legend()
    plt.tight_layout()
    fig1.savefig(f'{save_prefix}_1_目标分布.png')
    plt.close(fig1)
    print(f"✅ 目标分布图片已保存：{save_prefix}_1_目标分布.png")  # 保存成功提示
    
    # 图2: 特征相关性热力图
    fig2, ax2 = plt.subplots(figsize=(10, 8))
    corr_matrix = df.corr()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, 
                annot_kws={'size': 8}, ax=ax2)
    ax2.set_title(f'{name} - 特征相关性矩阵')
    plt.tight_layout()
    fig2.savefig(f'{save_prefix}_2_相关性矩阵.png')
    plt.close(fig2)
    print(f"✅ 相关性矩阵图片已保存：{save_prefix}_2_相关性矩阵.png")
    
    # 图3: 特征与目标变量相关性条形图
    fig3, ax3 = plt.subplots(figsize=(8, 6))
    feature_corr = corr_matrix['RASTERVALU'].drop('RASTERVALU').sort_values(ascending=True)
    colors = ['green' if x > 0 else 'red' for x in feature_corr]
    ax3.barh(range(len(feature_corr)), feature_corr, color=colors, alpha=0.7)
    ax3.set_yticks(range(len(feature_corr)))
    ax3.set_yticklabels(feature_corr.index, fontsize=9)
    ax3.set_xlabel('与RASTERVALU的相关系数')
    ax3.set_title(f'{name} - 特征与目标变量相关性')
    ax3.axvline(0, color='black', linewidth=0.5)
    plt.tight_layout()
    fig3.savefig(f'{save_prefix}_3_特征相关性.png')
    plt.close(fig3)
    print(f"✅ 特征相关性图片已保存：{save_prefix}_3_特征相关性.png")
    
    # 图4: Top4特征散点图
    fig4, axes4 = plt.subplots(2, 2, figsize=(10, 8))
    axes4 = axes4.flatten()
    top_features = feature_corr.abs().nlargest(4).index.tolist()
    for i, feat in enumerate(top_features):
        if i >= len(axes4): break
        axes4[i].scatter(X[feat], y, alpha=0.3, s=10, c='steelblue')
        axes4[i].set_xlabel(feat, fontsize=9)
        axes4[i].set_ylabel('RASTERVALU', fontsize=9)
        axes4[i].set_title(f'{feat} vs 目标变量', fontsize=10)
    # 隐藏多余子图
    for j in range(len(top_features), len(axes4)):
        axes4[j].set_visible(False)
    fig4.suptitle(f'{name} - Top4特征散点图', fontsize=12, y=1.02)
    plt.tight_layout()
    fig4.savefig(f'{save_prefix}_4_特征散点图.png')
    plt.close(fig4)
    print(f"✅ Top4特征散点图已保存：{save_prefix}_4_特征散点图.png")
    
    return X, y

def train_and_visualize(name, X, y, save_prefix):
    """模型训练+可视化（全基于LightGBM，适配旧版SHAP）"""
    print(f"\n{'='*60}")
    print(f"模型训练: {name}（LightGBM为核心可视化模型）")
    print(f"{'='*60}")
    
    # 数据分割与标准化（固定随机种子，确保结果可复现）
    X_shuffled, y_shuffled = shuffle(X, y, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(
        X_shuffled, y_shuffled, test_size=0.2, random_state=42
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    # 验证集分割（用于LGB早停）
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_scaled, y_train, test_size=0.2, random_state=42
    )
    
    # 模型训练（保留3个模型，但可视化仅用LightGBM）
    models = {}
    # 1. Random Forest
    rf = RandomForestRegressor(
        n_estimators=150, max_depth=10, min_samples_split=10,
        min_samples_leaf=8, max_features=0.6, random_state=42, n_jobs=-1
    )
    rf.fit(X_train_scaled, y_train)
    models['Random Forest'] = rf
    
    # 2. XGBoost
    xgb_model = xgb.XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.7, reg_alpha=5, reg_lambda=10,
        min_child_weight=8, early_stopping_rounds=30, random_state=42, n_jobs=-1
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    models['XGBoost'] = xgb_model
    
    # 3. LightGBM（核心模型，重点优化）
    lgb_model = lgb.LGBMRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.03,
        num_leaves=25, subsample=0.8, colsample_bytree=0.7,
        reg_alpha=5, reg_lambda=10, min_child_samples=25,
        random_state=42, n_jobs=-1, verbose=-1
    )
    lgb_model.fit(
        X_tr, y_tr, 
        eval_set=[(X_val, y_val)], 
        callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(False)]
    )
    models['LightGBM'] = lgb_model
    print(f"✅ LightGBM训练完成，最佳迭代次数：{lgb_model.best_iteration_}")
    
    # 模型性能评估（突出LightGBM）
    results = []
    predictions = {}
    print(f"\n{'模型':<15} {'训练R2':<10} {'测试R2':<10} {'差距':<10} {'MSE'}")
    print("-"*55)
    for model_name, model in models.items():
        y_train_pred = model.predict(X_train_scaled)
        y_test_pred = model.predict(X_test_scaled)
        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)
        gap = train_r2 - test_r2
        mse = mean_squared_error(y_test, y_test_pred)
        # 标记LightGBM（拟合最好的模型）
        mark = "⭐" if model_name == "LightGBM" else ""
        print(f"{model_name:<15} {train_r2:<10.4f} {test_r2:<10.4f} {gap:<10.4f} {mse:.2f} {mark}")
        results.append({
            'model': model_name, 'train_r2': train_r2, 'test_r2': test_r2, 'gap': gap, 'mse': mse
        })
        predictions[model_name] = y_test_pred
    
    # 图：模型拟合图（保留3个模型对比）
    fig_fit, axes_fit = plt.subplots(1, 3, figsize=(15, 5))
    for i, (model_name, y_pred) in enumerate(predictions.items()):
        ax = axes_fit[i]
        r2 = [r['test_r2'] for r in results if r['model'] == model_name][0]
        gap = [r['gap'] for r in results if r['model'] == model_name][0]
        ax.scatter(y_test, y_pred, alpha=0.5, s=20, c='steelblue')
        # 绘制参考线（y=x）
        min_val, max_val = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='理想拟合线')
        ax.set_xlabel('实际值', fontsize=9)
        ax.set_ylabel('预测值', fontsize=9)
        # 突出LightGBM
        title_color = 'darkred' if model_name == "LightGBM" else 'black'
        ax.set_title(f'{model_name}\nR2={r2:.4f}, Gap={gap:.4f}', fontsize=10, color=title_color)
        ax.legend(fontsize=8)
    plt.tight_layout()
    fig_fit.savefig(f'{save_prefix}_拟合图.png')
    plt.close(fig_fit)
    print(f"✅ 模型拟合图已保存：{save_prefix}_拟合图.png")
    
    # -------------------------- LightGBM-SHAP依赖图 --------------------------
    print(f"\n🔍 基于LightGBM执行SHAP依赖分析...")
    # 用LightGBM模型创建SHAP解释器
    explainer = shap.TreeExplainer(models['LightGBM'])
    # 计算SHAP值（处理多输出情况）
    try:
        shap_values = explainer.shap_values(X_test_scaled)
        # 如果是列表（多分类/多输出），取第一维（回归任务通常是单输出）
        if isinstance(shap_values, list):
            shap_values_array = shap_values[0]
            print(f"⚠️ SHAP值为列表格式，自动取第1维（回归任务默认）")
        else:
            shap_values_array = shap_values
        print(f"✅ SHAP值计算完成，维度：{shap_values_array.shape}")
    except Exception as e:
        raise RuntimeError(f"SHAP值计算失败：{str(e)}")
    
    features = X.columns
    # 过滤FID（无意义特征，不参与可视化）
    plot_features = [f for f in features if f != 'FID']
    plot_count = len(plot_features)
    if plot_count == 0:
        print("⚠️ 无有效特征可绘制SHAP依赖图（全部为FID）")
    else:
        # 计算子图行数（3列布局）
        rows = (plot_count + 2) // 3
        fig_shap_dep, axes_shap_dep = plt.subplots(rows, 3, figsize=(15, 4 * rows))  # 调整子图高度，避免拥挤
        axes_shap_dep = axes_shap_dep.flatten() if rows > 0 else []
        
        for plot_idx, feature in enumerate(plot_features):
            if plot_idx >= len(axes_shap_dep): break
            ax = axes_shap_dep[plot_idx]
            # 获取特征在原始X中的索引
            feat_idx = list(features).index(feature)
            # 用原始特征值（未标准化）绘图，更易解读
            feature_values = X_test[feature].values
            shap_for_feat = shap_values_array[:, feat_idx]
            
            # 绘制散点图
            ax.scatter(feature_values, shap_for_feat, alpha=0.3, s=15, color='steelblue')
            # 多项式拟合（动态调整阶数，避免过拟合）
            degree = min(5, plot_idx % 6 + 2)  # 限制最大阶数为5，避免拟合异常
            lx, ly = curve_poly(feature_values, shap_for_feat, degree)
            if len(lx) > 1:
                ax.plot(lx, ly, color='crimson', linewidth=2, label=f'{degree}阶拟合')
            
            ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, label='无影响线')
            ax.set_title(f'{feature}', fontsize=11)
            ax.set_xlabel('特征原始值', fontsize=9)
            ax.set_ylabel('SHAP值（影响程度）', fontsize=9)
            ax.legend(loc='best', fontsize=8)
        
        # 隐藏多余子图
        for j in range(plot_count, len(axes_shap_dep)):
            axes_shap_dep[j].set_visible(False)
        
        fig_shap_dep.suptitle(f'{name} - LightGBM-SHAP特征依赖分析', fontsize=14, y=1.02)
        plt.tight_layout()
        fig_shap_dep.savefig(f'{save_prefix}_LightGBM_SHAP依赖图.png')
        plt.close(fig_shap_dep)
        print(f"✅ LightGBM-SHAP依赖图已保存：{save_prefix}_LightGBM_SHAP依赖图.png")

    # -------------------------- LightGBM-特征重要性图 --------------------------
    print(f"\n🔍 基于LightGBM绘制特征重要性图...")
    # 提取3个模型的特征重要性，但按LightGBM排序
    importance_df = pd.DataFrame({
        '特征名称': features,
        'Random Forest': models['Random Forest'].feature_importances_,
        'XGBoost': models['XGBoost'].feature_importances_,
        'LightGBM': models['LightGBM'].feature_importances_
    })
    # 过滤FID + 按LightGBM重要性降序排序（最重要在最上方）
    importance_df = importance_df[importance_df['特征名称'] != 'FID'].sort_values(
        by='LightGBM', ascending=False
    )
    if len(importance_df) == 0:
        print("⚠️ 无有效特征可绘制重要性图（全部为FID）")
    else:
        fig_import, ax_import = plt.subplots(figsize=(12, 6 + len(importance_df)*0.3))  # 动态调整高度，避免特征名被截断
        x = np.arange(len(importance_df))
        width = 0.25  # 柱状图宽度
        
        # 绘制3个模型的重要性对比（LightGBM用深色突出）
        ax_import.barh(x - width, importance_df['Random Forest'], width, 
                      label='Random Forest', alpha=0.7, color='lightblue')
        ax_import.barh(x, importance_df['XGBoost'], width, 
                      label='XGBoost', alpha=0.7, color='orange')
        ax_import.barh(x + width, importance_df['LightGBM'], width, 
                      label='LightGBM（拟合最优）', alpha=0.8, color='darkred')
        
        # 标签与图例优化
        ax_import.set_yticks(x)
        ax_import.set_yticklabels(importance_df['特征名称'], fontsize=10)
        ax_import.set_xlabel('特征重要性（数值越大，对预测影响越强）', fontsize=11)
        ax_import.set_title(f'{name} - 特征重要性对比（按LightGBM重要性降序）', fontsize=13)
        ax_import.legend(loc='lower right', fontsize=9)
        # 添加网格线，便于读取数值
        ax_import.grid(axis='x', alpha=0.3, linestyle='-')
        
        plt.tight_layout()
        fig_import.savefig(f'{save_prefix}_LightGBM_特征重要性对比.png')
        plt.close(fig_import)
        print(f"✅ LightGBM-特征重要性图已保存：{save_prefix}_LightGBM_特征重要性对比.png")

    # -------------------------- 核心修复：LightGBM-SHAP摘要散点图（移除ax参数） --------------------------
    print(f"\n🔍 基于LightGBM绘制SHAP摘要散点图...")
    if plot_count == 0:
        print("⚠️ 无有效特征可绘制SHAP摘要图（全部为FID）")
    else:
        # 过滤FID对应的SHAP值和特征名
        if 'FID' in features:
            fid_idx = list(features).index('FID')
            shap_values_filtered = np.delete(shap_values_array, fid_idx, axis=1)
            features_filtered = [f for f in features if f != 'FID']
            X_test_filtered = X_test.drop(columns=['FID'])
        else:
            shap_values_filtered = shap_values_array
            features_filtered = features
            X_test_filtered = X_test
        
        # 【修复】移除不兼容的'ax'参数，旧版SHAP自动在当前画布绘图
        plt.figure(figsize=(10, 6 + len(features_filtered)*0.2))  # 动态调整高度
        shap.summary_plot(
            shap_values_filtered, 
            features=X_test_filtered,
            feature_names=features_filtered,
            plot_type="dot",  # 散点模式（核心需求）
            show=False,       # 不弹出预览，仅保存
            plot_size=None,
            color_bar_label="特征原始值大小"  # 颜色条标签
        )
        # 调整标题（用plt.title替代ax.set_title，适配旧版）
        plt.title(f'{name} - LightGBM-SHAP特征影响摘要散点图', fontsize=13, pad=20)
        plt.tight_layout()  # 自动调整布局，避免标签截断
        plt.savefig(f'{save_prefix}_LightGBM_SHAP摘要散点图.png')
        plt.close()  # 关闭画布，释放内存
        print(f"✅ LightGBM-SHAP摘要散点图已保存：{save_prefix}_LightGBM_SHAP摘要散点图.png")

    return results, y_test, predictions

def plot_comparison(all_results):
    """跨道路类型模型对比图（突出LightGBM）"""
    datasets = list(all_results.keys())
    if len(datasets) == 0:
        print("⚠️ 无数据可用于跨道路类型对比")
        return
    
    models = ['Random Forest', 'XGBoost', 'LightGBM']
    # 图1: 测试R2对比（突出LightGBM）
    fig_comp_r2, ax_comp_r2 = plt.subplots(figsize=(8, 6))
    x = np.arange(len(datasets))
    width = 0.25
    colors = ['lightblue', 'orange', 'darkred']  # LightGBM用深色
    labels = ['Random Forest', 'XGBoost', 'LightGBM（拟合最优）']
    
    for i, (model, color, label) in enumerate(zip(models, colors, labels)):
        test_r2s = [all_results[d]['results'][i]['test_r2'] for d in datasets]
        bars = ax_comp_r2.bar(x + i*width, test_r2s, width, label=label, alpha=0.8, color=color)
        # 在柱子上添加数值标签
        for bar, r2 in zip(bars, test_r2s):
            height = bar.get_height()
            ax_comp_r2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                           f'{r2:.3f}', ha='center', va='bottom', fontsize=8)
    
    ax_comp_r2.set_xticks(x + width)
    ax_comp_r2.set_xticklabels(datasets, fontsize=10)
    ax_comp_r2.set_ylabel('测试R2（越高拟合越好）', fontsize=11)
    ax_comp_r2.set_title('不同道路类型 - 模型测试R2对比（LightGBM最优）', fontsize=12)
    ax_comp_r2.legend(fontsize=9)
    ax_comp_r2.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fig_comp_r2.savefig('模型对比_测试R2.png')
    plt.close(fig_comp_r2)
    print(f"\n✅ 跨道路类型R2对比图已保存：模型对比_测试R2.png")
    
    # 图2: 过拟合差距对比
    fig_comp_gap, ax_comp_gap = plt.subplots(figsize=(8, 6))
    for i, (model, color, label) in enumerate(zip(models, colors, labels)):
        gaps = [all_results[d]['results'][i]['gap'] for d in datasets]
        bars = ax_comp_gap.bar(x + i*width, gaps, width, label=label, alpha=0.8, color=color)
        # 添加数值标签
        for bar, gap in zip(bars, gaps):
            height = bar.get_height()
            ax_comp_gap.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                            f'{gap:.3f}', ha='center', va='bottom', fontsize=8)
    
    ax_comp_gap.set_xticks(x + width)
    ax_comp_gap.set_xticklabels(datasets, fontsize=10)
    ax_comp_gap.set_ylabel('过拟合差距（训练R2-测试R2，越小越好）', fontsize=11)
    ax_comp_gap.set_title('不同道路类型 - 模型过拟合差距对比', fontsize=12)
    ax_comp_gap.legend(fontsize=9)
    ax_comp_gap.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fig_comp_gap.savefig('模型对比_过拟合差距.png')
    plt.close(fig_comp_gap)
    print(f"✅ 跨道路类型过拟合差距图已保存：模型对比_过拟合差距.png")

if __name__ == "__main__":
    # 1. 配置数据（确保文件名与实际一致，相对路径）
    datasets = [
        ("支路", "支路自变量与因变量0112.xlsx", ['FID', '行人数量', '骑行者数量', '机动车数量', '非机动车数量', '绿视率', '道路机动化程度', '人行空间', '街道复杂化程度', '建成界面围合度']),
        ("主干道", "主干道自变量与因变量0112.xlsx", ['FID', '行人数量', '骑行者数量', '机动车数量', '非机动车数量', '绿视率', '围合度', '道路机动化程度', '建成界面围合度']),
        ("次干道", "次干道自变量与因变量0112.xlsx", ['FID', '行人数量', '骑行者数量', '机动车数量', '非机动车数量', '绿视率', '道路机动化程度', '人行空间', '街道复杂化程度', '建成界面围合度'])
    ]
    all_results = {}
    
    # 2. 检查当前工作目录和文件（关键：确保数据在当前目录）
    import os
    current_dir = os.getcwd()
    print(f"当前工作目录：{current_dir}")
    print(f"目录下的文件：{os.listdir(current_dir)}")
    # 验证数据文件是否存在
    for name, file_path, _ in datasets:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"数据文件不存在：{file_path}，请确认文件在当前目录（{current_dir}）")
    print(f"\n✅ 所有数据文件已找到，开始执行...")
    
    # 3. 逐数据集执行：探索→训练→可视化
    for name, file_path, target_features in datasets:
        # 读取数据 + 过滤无效列（Unnamed列）
        df = pd.read_excel(file_path)
        df = df.loc[:, ~df.columns.str.contains('Unnamed', na=False)]
        print(f"\n{'='*40}")
        print(f"开始处理：{name}")
        print(f"数据过滤后列名：{list(df.columns)}")
        
        # 选择目标特征（只保留预设的视觉特征）
        available_features = [f for f in target_features if f in df.columns]
        if len(available_features) == 0:
            raise ValueError(f"{name}数据中未找到任何目标特征，请检查特征列名")
        # 确保包含目标变量
        if 'RASTERVALU' not in df.columns:
            raise ValueError(f"{name}数据中未找到目标变量'RASTERVALU'")
        df_selected = df[available_features + ['RASTERVALU']]
        
        # 数据探索 + 模型训练可视化
        X, y = explore_data(name, df_selected, name)
        results, y_t, preds = train_and_visualize(name, X, y, name)
        all_results[name] = {'results': results}
    
    # 4. 跨道路类型对比
    plot_comparison(all_results)
    
    # 5. 最终性能汇总（突出LightGBM）
    print(f"\n{'#'*80}")
    print("# 最终模型性能汇总（LightGBM拟合最优）")
    print(f"{'#'*80}")
    print(f"\n{'道路类型':<8} {'模型':<20} {'训练R2':<12} {'测试R2':<12} {'过拟合差距':<12}")
    print("="*75)
    for name, data in all_results.items():
        for r in data['results']:
            mark = "⭐" if r['model'] == "LightGBM" else ""
            print(f"{name:<8} {r['model']:<20} {r['train_r2']:<12.4f} {r['test_r2']:<12.4f} {r['gap']:<12.4f} {mark}")
    print("="*75)
    print(f"\n🎉 所有任务完成！图片已保存到当前工作目录：{current_dir}")

当前工作目录：D:\9014-V2(1)\9014-V2
目录下的文件：['.idea', '.ipynb_checkpoints', 'runnew_lyh.ipynb', 'run_ml.ipynb', 'run_ml.py', '~$主干道自变量与因变量0112.xlsx', '~$支路自变量与因变量0112.xlsx', '~$次干道自变量与因变量0112.xlsx', '主干道_1_目标分布.png', '主干道_2_相关性矩阵.png', '主干道_3_特征相关性.png', '主干道_4_特征散点图.png', '主干道_SHAP依赖图.png', '主干道_SHAP摘要散点图.png', '主干道_拟合图.png', '主干道_特征重要性对比.png', '主干道自变量与因变量0112.xlsx', '主干道自变量与因变量（人工围合度）0104.xlsx', '支路_1_目标分布.png', '支路_2_相关性矩阵.png', '支路_3_特征相关性.png', '支路_4_特征散点图.png', '支路_LightGBM_SHAP依赖图.png', '支路_LightGBM_特征重要性对比.png', '支路_SHAP依赖图.png', '支路_SHAP摘要散点图.png', '支路_拟合图.png', '支路_特征重要性对比.png', '支路自变量与因变量0112.xlsx', '支路自变量与因变量（人工围合度）0104.xlsx', '模型对比_1_测试R2.png', '模型对比_2_过拟合差距.png', '模型对比_3_训练vs测试.png', '模型对比_测试R2.png', '模型对比_过拟合差距.png', '次干道_1_目标分布.png', '次干道_2_相关性矩阵.png', '次干道_3_特征相关性.png', '次干道_4_特征散点图.png', '次干道_SHAP依赖图.png', '次干道_SHAP摘要散点图.png', '次干道_拟合图.png', '次干道_特征重要性对比.png', '次干道自变量与因变量0112.xlsx', '次干道自变量与因变量（人工围合度）0104.xlsx']

✅ 所有数据文件已找到，开始执行...

开始处理：支路
数据过滤后列名：['FID', '行人数量', '骑行者数量', '